In [ ]:
# ===============================
# Cell 1: Imports and Load Data
# ===============================
import pandas as pd
import numpy as np
import shap
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import (f1_score, precision_score, recall_score, roc_auc_score,
                             confusion_matrix, make_scorer)
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, cross_val_predict
from sklearn.base import clone

# Load data
print("📂 Loading data...")
df = pd.read_csv("../data/mcs_sh_sample_Feb_28_preprocessed.csv")

# Encode target
df["suicide_17y"] = df["suicide_17y"].map({"no": 0, "yes": 1})

# Separate features and target
X = df.drop(columns=["suicide_17y"])
y = df["suicide_17y"].astype(int)

# Drop ID if present
if "id" in X.columns:
    X = X.drop(columns=["id"])

# Convert "True"/"False" strings to actual boolean, then float
X = X.replace({"True": True, "False": False})
X = X.astype(float)

print(X.dtypes.value_counts())

# Define CV folds
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
print("✅ Data loaded and CV defined.")

In [ ]:
# ===============================
# Cell 2: Nested CV with SHAP aggregation
# ===============================

import numpy as np
import shap
from sklearn.metrics import (
    f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix
)
from sklearn.model_selection import GridSearchCV

def find_best_threshold(y_true, y_probs, metric=f1_score):
    best_thresh = 0.5
    best_score = -1
    thresholds = np.linspace(0.0, 1.0, 101)
    for t in thresholds:
        preds = (y_probs >= t).astype(int)
        try:
            score = metric(y_true.astype(int), preds)
        except:
            continue
        if score > best_score:
            best_thresh = t
            best_score = score
    return best_thresh, best_score


def nested_cv_evaluation(X, y, model, param_grid, inner_cv, outer_cv, primary_metric='f1'):

    def npv_score(y_true, y_pred):
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        return tn / (tn + fn) if (tn + fn) > 0 else 0

    def specificity_score(y_true, y_pred):
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        return tn / (tn + fp) if (tn + fp) > 0 else 0

    scoring_funcs = {
        'F1': f1_score,
        'Weighted F1': lambda y, p: f1_score(y, p, average='weighted'),
        'Precision': precision_score,
        'Recall': recall_score,
        'Specificity': specificity_score,
        'NPV': npv_score,
        'AUROC': roc_auc_score
    }

    results_opt = {metric: [] for metric in scoring_funcs}
    results_default = {metric: [] for metric in scoring_funcs}
    thresholds = []
    all_shap_importances = []
    all_shap_values = []
    all_shap_X = []

    for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), start=1):
        print(f"\n🌊 Outer Fold {fold_idx}")
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # === Inner CV (Hyperparameter tuning) ===
        grid_search = GridSearchCV(
            estimator=model,
            param_grid=param_grid,
            cv=inner_cv,
            scoring=primary_metric,
            n_jobs=-1,
            verbose=0
        )
        grid_search.fit(X_train, y_train)
        best_model = grid_search.best_estimator_
        is_mlp = "MLPClassifier" in str(type(best_model))

        # === SHAP feature importances (safe) ===
        try:
            shap_values = None
            if "XGB" in str(type(best_model)):
                explainer = shap.TreeExplainer(best_model)
                shap_values = explainer.shap_values(X_test)
                # xgboost binary sometimes returns list; normalize to array
                if isinstance(shap_values, list):
                    shap_values = shap_values[1] if len(shap_values) > 1 else shap_values[0]
            elif "LogisticRegression" in str(type(best_model)):
                explainer = shap.LinearExplainer(best_model, X_train)
                shap_values = explainer.shap_values(X_test)
            elif "MLPClassifier" in str(type(best_model)):
                # KernelExplainer fallback (slow but bounded + safe)
                try:
                    background = shap.sample(X_train, min(100, len(X_train)), random_state=42)
                    explainer = shap.KernelExplainer(best_model.predict_proba, background)
                    # small subsample for speed + stability
                    sv = explainer.shap_values(X_test[:min(80, len(X_test))])
                    # for binary, pick class 1; some SHAP versions return list
                    shap_values = sv[1] if isinstance(sv, list) else sv
                except Exception as e:
                    print(f"⚠️ SHAP failed for MLP fold ({e}), skipping explainability.")
                    shap_values = None
        
            if shap_values is not None:
                shap_values = np.array(shap_values)
                # Ensure 2D: (n_samples, n_features)
                if shap_values.ndim == 3:
                    # some explainers return (n_classes, n_samples, n_features)
                    shap_values = shap_values[1] if shap_values.shape[0] > 1 else shap_values[0]
                if shap_values.ndim == 2:
                    # Sometimes SHAP returns transposed arrays for MLP: (n_features, n_samples)
                    if shap_values.shape[0] == X_train.shape[1] and shap_values.shape[1] != X_train.shape[1]:
                        print(f"↩️ Detected transposed SHAP output (shape {shap_values.shape}), fixing...")
                        shap_values = shap_values.T
                
                    if shap_values.shape[1] == X_train.shape[1]:
                        fold_importance = np.abs(shap_values).mean(axis=0)
                        all_shap_importances.append(fold_importance)
                            # Save actual fold-level SHAP values for matched plots

                        if "MLPClassifier" in str(type(best_model)):
                            X_shap_fold = X_test.iloc[:shap_values.shape[0]].copy()
                        else:                    
                            X_shap_fold = X_test.copy()                    
                        all_shap_values.append(shap_values)                    
                        all_shap_X.append(X_shap_fold)
                    
                    else:
                        print(f"ℹ️ Skipping SHAP for this fold (shape {shap_values.shape}), "
                              f"expected (*, {X_train.shape[1]}).")
        except Exception as e:
            print(f"⚠️ Skipping SHAP for this fold due to error: {e}")

        # === Predictions ===
        y_proba_test = best_model.predict_proba(X_test)[:, 1]
        
        # Default threshold (0.5) evaluated on OUTER test
        y_pred_default = (y_proba_test >= 0.5).astype(int)
        for metric, func in scoring_funcs.items():
            score = func(y_test, y_proba_test) if metric == "AUROC" else func(y_test, y_pred_default)
            results_default[metric].append(score)
        
        # === Choose threshold using INNER CV out-of-fold predictions on OUTER train ===
        y_proba_oof = cross_val_predict(
            clone(best_model),
            X_train,
            y_train,
            cv=inner_cv,
            method="predict_proba",
            n_jobs=-1
        )[:, 1]
        
        best_thresh, _ = find_best_threshold(y_train, y_proba_oof)
        thresholds.append(best_thresh)
        
        # Optimized threshold evaluated on OUTER test
        y_pred_opt = (y_proba_test >= best_thresh).astype(int)
        for metric, func in scoring_funcs.items():
            score = func(y_test, y_proba_test) if metric == "AUROC" else func(y_test, y_pred_opt)
            results_opt[metric].append(score)
        
        print(f"✅ Best threshold from inner CV OOF for fold {fold_idx}: {best_thresh:.2f}")

    # === Aggregate SHAP importances across folds (safe) ===
    try:
        if len(all_shap_importances) > 0:
            A = np.vstack(all_shap_importances)          # shape: (n_good_folds, n_features)
            mean_shap_importance = A.mean(axis=0)        # (n_features,)
            shap_summary_df = (
                pd.DataFrame({"Feature": X.columns, "MeanAbsSHAP": mean_shap_importance})
                .sort_values("MeanAbsSHAP", ascending=False)
                .reset_index(drop=True)
            )
        else:
            shap_summary_df = None
    except Exception as e:
        print(f"⚠️ SHAP aggregation failed ({e}); continuing without SHAP.")
        shap_summary_df = None

    try:
        if len(all_shap_values) > 0:
            shap_values_cv = np.vstack(all_shap_values)
            X_shap_cv = pd.concat(all_shap_X, axis=0)
        else:
            shap_values_cv = None
            X_shap_cv = None
    except Exception as e:
        print(f"⚠️ SHAP value aggregation for plots failed ({e}).")
        shap_values_cv = None
        X_shap_cv = None

    return results_opt, thresholds, results_default, shap_summary_df, shap_values_cv, X_shap_cv

In [ ]:
# ===============================
# Cell 3: Define Models + Grids
# ===============================

xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb_grid = {
    'n_estimators': [100],
    'max_depth': [3, 5],
    'learning_rate': [0.1],
    'scale_pos_weight': [1, (y == 0).sum() / (y == 1).sum()]
}

lr_model = LogisticRegression(solver='liblinear', class_weight='balanced')
lr_grid = {
    'C': [0.1, 1, 10]
}

mlp_model = MLPClassifier(random_state=42, max_iter=500)
mlp_grid = {
    'hidden_layer_sizes': [(64,), (64, 32)],
    'alpha': [0.0001, 0.01]
}

In [ ]:
# ===========================================
# Cell 4: Nested Cross-Validation + Unified Top-5 SHAP Table
# ===========================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


metrics = [
    ('AUCPR', 'average_precision'),
    ('Balanced Acc', 'balanced_accuracy'),
    ('F1 Weighted', 'f1_weighted'),
    ('F1 (macro)', 'f1')
]

models = {
    "XGBoost": (xgb_model, xgb_grid),
    "LogReg": (lr_model, lr_grid),
    "MLP": (mlp_model, mlp_grid)
}

results_summary = []
top_features_summary = []   # rows for the global shap comparison table

for model_name, (model, grid) in models.items():
    for metric_name, metric_key in metrics:
        print(f"\n🚀 Running {model_name} optimized for {metric_name} ...")

        try:
            # === Run nested CV with SHAP aggregation ===
            results_opt, thresholds, results_default, shap_summary, shap_values_cv, X_shap_cv = nested_cv_evaluation(
                X, y, model, grid, inner_cv, outer_cv, primary_metric=metric_key
            )

            # --- store performance summary ---
            results_summary.append({
                "Model": f"{model_name} ({metric_name})",
                "F1": np.mean(results_opt['F1']),
                "Weighted F1": np.mean(results_opt['Weighted F1']),
                "Precision": np.mean(results_opt['Precision']),
                "Recall": np.mean(results_opt['Recall']),
                "Specificity": np.mean(results_opt['Specificity']),
                "NPV": np.mean(results_opt['NPV']),
                "AUROC": np.mean(results_opt['AUROC']),
                "Threshold": np.mean(thresholds)
            })

            # --- extract top-5 SHAP features ---
            if shap_summary is not None:
                top5 = shap_summary.head(5)
                row = {"Model": f"{model_name} ({metric_name})"}
                for i, (feat, val) in enumerate(zip(top5["Feature"], top5["MeanAbsSHAP"]), start=1):
                    row[f"Top {i}"] = feat
                    row[f"SHAP {i}"] = round(val, 6)
                top_features_summary.append(row)
            else:
                print(f"⚠️ No SHAP values computed for {model_name} ({metric_name})")

        except Exception as e:
            print(f"⚠️ Skipping {model_name} ({metric_name}) due to error: {e}")
            continue
        
        # === SHAP summary plot from outer-fold SHAP values ===
        
        if shap_summary is not None and shap_values_cv is not None and X_shap_cv is not None:
            try:
                print(f"\n📊 Generating CV-based SHAP Summary Plot for {model_name} ({metric_name}) ...")
        
                safe_metric = metric_name.replace(" ", "_").replace("(", "").replace(")", "")
        
                shap.summary_plot(
                    shap_values_cv,
                    X_shap_cv,
                    plot_type="dot",
                    show=False
                )
        
                plt.title(f"CV-based SHAP Summary Plot - {model_name} ({metric_name})")
                plt.tight_layout()
                plt.savefig(f"shap_summary_CV_{model_name}_{safe_metric}.png", dpi=300, bbox_inches="tight")
                plt.show()
        
            except Exception as e:
                print(f"⚠️ Could not generate CV-based SHAP summary plot: {e}")

# ---- Convert to DataFrames ----
results_df = pd.DataFrame(results_summary)
top5_df = pd.DataFrame(top_features_summary)

# ---- Save Outputs ----
results_df.to_csv("results_ML/nestedCV_results_summary_sh_sample.csv", index=False)
top5_df.to_csv("results_ML/nestedCV_top5_features_sh_sample.csv", index=False)

print("\n✅ Nested CV and SHAP computation complete!")
print("📊 Performance summary → nestedCV_results_summary_sh_sample.csv")
print("🌟 Top-5 SHAP summary → nestedCV_top5_features_sh_sample.csv")

display(results_df)
display(top5_df)

In [ ]:
import sklearn
print(sklearn.__version__)

In [ ]:
# ===============================
# Cell 5: SHAP Force Plot (Single Prediction)
# ===============================
import shap
import matplotlib.pyplot as plt

# === 1. Choose a sample index (change this as needed) ===
sample_index = 0
X_sample = X.iloc[[sample_index]]

# === 2. Refit the best XGBoost model on full data (if not already done) ===
print("🔁 Re-training XGBoost model on full data...")
xgb_model.fit(X, y)

# === 3. Create SHAP explainer and compute SHAP values ===
explainer = shap.Explainer(xgb_model, X)
shap_values = explainer(X_sample)

# === 4. Generate force plot ===
print(f"📈 Generating SHAP force plot for sample index {sample_index}...")
shap.plots.force(shap_values[0])

In [ ]:
pip install mljar-supervised

In [ ]:
#Compare performance with Auto ML
# ===============================
# AutoML with 80/20 Split
# ===============================
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix
from supervised.automl import AutoML

# === Step 1: Load preprocessed data ===
df = pd.read_csv("final_preprocessed_dataset.csv")
X = df.drop(columns=["suicide_17y"])
y = df["suicide_17y"].astype(str)  # required as string for mljar

# === Step 2: Train/Test split ===
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# === Step 3: Fill missing values (median imputation) ===
X_train = X_train.fillna(X_train.median(numeric_only=True))
X_test = X_test.fillna(X_train.median(numeric_only=True))  # use train medians

# === Step 4: Train AutoML ===
automl = AutoML(
    mode="Compete",  # Or "Perform" for speed
    eval_metric="f1",
    results_path="automl_suicide_run",
    algorithms=["Xgboost", "Random Forest", "LightGBM", "CatBoost", "Neural Network", "Linear"],
    validation_strategy={
        "validation_type": "split",
        "train_ratio": 0.8,
        "stratify": True,
        "shuffle": True,
    }
)
automl.fit(X_train, y_train)

# === Step 5: Predict Probabilities ===
y_pred_proba = automl.predict_proba(X_test)[:, 1]

# === Convert y_test to binary (0/1) ===
y_test_binary = y_test.map({'no': 0, 'yes': 1}).astype(int)

# === Step 6: Find best threshold ===
def find_best_threshold(y_true, y_probs):
    thresholds = np.arange(0.05, 0.95, 0.01)
    best_thresh, best_score = 0.5, -1
    for t in thresholds:
        preds = (y_probs >= t).astype(int)
        score = f1_score(y_true, preds)
        if score > best_score:
            best_thresh = t
            best_score = score
    return best_thresh

best_thresh = find_best_threshold(y_test_binary, y_pred_proba)
y_pred_opt = (y_pred_proba >= best_thresh).astype(int)

# === Step 7: Metrics ===
def npv_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fn) if (tn + fn) > 0 else 0

def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp) if (tn + fp) > 0 else 0

metrics = {
    "F1": f1_score(y_test_binary, y_pred_opt),
    "Precision": precision_score(y_test_binary, y_pred_opt),
    "Recall": recall_score(y_test_binary, y_pred_opt),
    "NPV": npv_score(y_test_binary, y_pred_opt),
    "AUROC": roc_auc_score(y_test_binary, y_pred_proba),
    "Specificity": specificity_score(y_test_binary, y_pred_opt),
}

print("\n📊 AutoML Test Set Results")
for m, v in metrics.items():
    print(f"{m}: {v:.3f}")
print(f"Optimal threshold: {best_thresh:.2f}")

In [ ]:
!pip install tabpfn

In [ ]:
# ============================================
# 🧠 TabPFN: Foundation Model (Test Set Only)
# ============================================
import pandas as pd
import numpy as np
from tabpfn import TabPFNClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix
import torch

# === 1. Load processed dataset ===
df = pd.read_csv("final_preprocessed_dataset.csv")
X = df.drop(columns=["suicide_17y"])
y = df["suicide_17y"].map({"no": 0, "yes": 1}).astype(int)

# === 2. Train/Test Split ===
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# === 3. Impute missing values ===
X_train = X_train.fillna(X_train.median(numeric_only=True))
X_test = X_test.fillna(X_train.median(numeric_only=True))  # use train medians

# === 4. Fit TabPFN ===
clf = TabPFNClassifier(
    device="cuda" if torch.cuda.is_available() else "cpu",
    ignore_pretraining_limits=True
)
clf.fit(X_train, y_train)

# === 5. Predict probabilities ===
y_proba = clf.predict_proba(X_test)[:, 1]

# === 6. Find optimal threshold ===
best_thresh, _ = find_best_threshold(y_test, y_proba)
y_pred = (y_proba >= best_thresh).astype(int)

# === 7. Compute metrics ===
def npv_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fn) if (tn + fn) > 0 else 0

def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp) if (tn + fp) > 0 else 0

f1       = f1_score(y_test, y_pred)
precision= precision_score(y_test, y_pred)
recall   = recall_score(y_test, y_pred)
npv      = npv_score(y_test, y_pred)
auroc    = roc_auc_score(y_test, y_proba)
specificity = specificity_score(y_test_binary, y_pred_opt),

# === 8. Display results ===
print("\n📊 TabPFN Test Set Results")
print(f"F1: {f1:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"NPV: {npv:.3f}")
print(f"AUROC: {auroc:.3f}")
print(f"Optimal threshold: {best_thresh:.2f}")

# === 9. Append to results_df comparison table ===
new_row = {
    "Metric": "TabPFN (Test Set)",
    "F1":       f"{f1:.3f} ({best_thresh:.2f})",
    "Precision":f"{precision:.3f} ({best_thresh:.2f})",
    "Recall":   f"{recall:.3f} ({best_thresh:.2f})",
    "NPV":      f"{npv:.3f} ({best_thresh:.2f})",
    "AUROC":    f"{auroc:.3f}"
}

# If results_df does not exist yet, initialize it
if "results_df" not in locals():
    results_df = pd.DataFrame(columns=["Metric", "F1", "Precision", "Recall", "NPV", "AUROC"])

results_df = pd.concat([results_df, pd.DataFrame([new_row])], ignore_index=True)
display(results_df)

In [ ]:
import sys
sys.version